# OSAIMA intent model: LoRA fine-tune on a free GPU

Fine-tunes **Qwen2.5-0.5B-Instruct** into OSAIMA's intent parser (request -> compact intent JSON),
merges the adapter and exports **GGUF Q4_K_M** for `llama-server`.

* Runtime: GPU (T4 on Colab free tier; any CUDA GPU works). About 10-15 min for 3 epochs on a T4.
* Inputs: `ai-core/intent-model/` (`train.py`, `fmt.py`, `data/train.jsonl`) and
  `ai-core/mcp-daemon/src/prompts/intent_fast.txt` (the short system prompt the daemon sends).
* Output: `osaima-intent-qwen0.5b-Q4_K_M.gguf` (~400 MB). Copy it to `~/.local/share/osaima/models/`
  and start it with `ai-core/llm/start-intent-llm.sh`.

The same `train.py` runs on a local CUDA machine: `python train.py` (add `--unsloth` if Unsloth is installed).

## 1. Get the code and data
Either clone the repo (set `REPO`; a private repo needs a token in the URL) or upload `intent-model.zip` made with
`cd ai-core && zip -r intent-model.zip intent-model/{train.py,fmt.py,export.sh,data} mcp-daemon/src/prompts/intent_fast.txt`.

In [ ]:
REPO = ""  # e.g. "https://<token>@github.com/<you>/osaima.git"; leave empty to upload a zip
import os, subprocess
!nvidia-smi --query-gpu=name,memory.total --format=csv
if REPO:
    !git clone --depth 1 -b feature/lua-window-manager {REPO} osaima
    os.chdir("osaima/ai-core/intent-model")
else:
    from google.colab import files
    up = files.upload()  # intent-model.zip
    !mkdir -p ai-core && unzip -o -q intent-model.zip -d ai-core
    os.chdir("ai-core/intent-model")
!ls && wc -l data/train.jsonl

## 2. Install
Unsloth is optional (faster); the plain `transformers` + `peft` path is the fallback.

In [ ]:
USE_UNSLOTH = True
!pip -q install peft gguf sentencepiece protobuf
if USE_UNSLOTH:
    try:
        !pip -q install unsloth
        import unsloth  # noqa: F401
    except Exception as e:
        print("Unsloth unavailable, using plain PEFT:", e)
        USE_UNSLOTH = False

## 3. Train (LoRA r=16 on all linear layers, 3 epochs, lr 2e-4, batch 16)

In [ ]:
flag = "--unsloth" if USE_UNSLOTH else ""
!python train.py --epochs 3 --batch 16 --out out/qwen0.5b {flag}
!cat out/qwen0.5b/train_info.json

If Unsloth fails on this runtime, rerun the cell above with `USE_UNSLOTH = False`.

For the 1.5B model: `!python train.py --model Qwen/Qwen2.5-1.5B-Instruct --out out/qwen1.5b --batch 8`.

## 4. Quick check on the held-out split (transformers, greedy)

In [ ]:
import json, torch, fmt
from transformers import AutoModelForCausalLM, AutoTokenizer
path = "out/qwen0.5b/merged"
tok = AutoTokenizer.from_pretrained(path)
model = AutoModelForCausalLM.from_pretrained(path, dtype=torch.float16).cuda().eval()
rows = [json.loads(l) for l in open("data/test.jsonl")][:150]
ok = 0
for r in rows:
    text = tok.apply_chat_template(fmt.messages(r["request"], r["now"]), tokenize=False, add_generation_prompt=True)
    ids = tok(text, return_tensors="pt").to("cuda")
    out = model.generate(**ids, max_new_tokens=200, do_sample=False)
    ans = tok.decode(out[0][ids["input_ids"].shape[1]:], skip_special_tokens=True)
    try:
        ok += json.loads(ans)["intent"] == r["target"]["intent"]
    except Exception:
        pass
print(f"intent accuracy on {len(rows)} held-out requests: {ok / len(rows):.1%}")

## 5. Export GGUF Q4_K_M (builds only `llama-quantize`)

In [ ]:
!git clone -q --depth 1 https://github.com/ggml-org/llama.cpp llama.cpp
!cmake -S llama.cpp -B llama.cpp/build -DGGML_CUDA=OFF -DLLAMA_CURL=OFF > /dev/null && cmake --build llama.cpp/build --target llama-quantize -j 4 > /dev/null
!LLAMA_QUANTIZE=llama.cpp/build/bin/llama-quantize PYTHON=python QUANT_THREADS=4 bash export.sh out/qwen0.5b osaima-intent-qwen0.5b

## 6. Download

In [ ]:
from google.colab import files
files.download("out/qwen0.5b/osaima-intent-qwen0.5b-Q4_K_M.gguf")

Then on the OSAIMA machine:

```bash
cp osaima-intent-qwen0.5b-Q4_K_M.gguf ~/.local/share/osaima/models/
ai-core/llm/start-intent-llm.sh &            # llama-server on 127.0.0.1:8081
cd ai-core/intent-model && python bench.py distilled   # accuracy + latency on data/test.jsonl
```